# Dental Diagnostic AI - End-to-End Colab Pipeline

Runs the full pipeline: Kaggle download -> preprocessing -> augmentation -> YOLO detection training -> U-Net segmentation training -> evaluation -> Grad-CAM demo -> deploy API + Streamlit app via ngrok.

**Set Runtime > Change runtime type > GPU (T4) before running.**

## 0. Clone repo and install dependencies

In [ ]:
!git clone https://github.com/<your-username>/dental-ai-diagnostics.git
%cd dental-ai-diagnostics
!pip install -q -r requirements.txt


## 1. Kaggle setup and dataset download
Upload your `kaggle.json` (from kaggle.com/settings > API > Create New Token) when prompted.

In [ ]:
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json


In [ ]:
!python data/kaggle_download.py --dataset reemsalahshehab/dental-datasetv6 --out /content/data --fix-yaml


## 2. Preprocessing (CLAHE + denoise + letterbox, labels remapped in lockstep)

In [ ]:
import os
for split in ['train', 'valid', 'test']:
    os.system(
        f'python data/preprocess.py '
        f'--src /content/data/{split}/images --dst /content/data_proc/{split}/images '
        f'--labels-src /content/data/{split}/labels --labels-dst /content/data_proc/{split}/labels '
        f'--size 1024'
    )
print('Preprocessing done.')


## 3. Augmentation (train split only)

In [ ]:
!python data/augment.py \
  --images /content/data_proc/train/images --labels /content/data_proc/train/labels \
  --out-images /content/data_aug/train/images --out-labels /content/data_aug/train/labels \
  --multiplier 3


In [ ]:
# point augmented train split back at the original data.yaml structure
import shutil, yaml
shutil.copytree('/content/data_proc/valid', '/content/data_aug/valid', dirs_exist_ok=True)
shutil.copytree('/content/data_proc/test', '/content/data_aug/test', dirs_exist_ok=True)
shutil.copy('/content/data/data.yaml', '/content/data_aug/data.yaml')


## 4. Stage 1 - Train YOLO tooth/finding detector

In [ ]:
!python models/detection/train.py \
  --data /content/data_aug/data.yaml --model yolov8s.pt \
  --epochs 60 --imgsz 640 --batch 16


## 5. Stage 2 - Train U-Net pathology segmentation (uses COCO masks)

In [ ]:
%cd models/segmentation
!python train.py \
  --images /content/data/COCO/COCO/train/images \
  --coco /content/data/COCO/COCO/annotations/train_coco.json \
  --val-images /content/data/COCO/COCO/valid/images \
  --val-coco /content/data/COCO/COCO/annotations/valid_coco.json \
  --epochs 40 --batch 8
%cd /content/dental-ai-diagnostics


## 6. Evaluation (fixed IoU matching, per-class metrics, confusion matrix)

In [ ]:
import sys; sys.path.append('.')
import supervision as sv
from ultralytics import YOLO
from evaluation.metrics import evaluate_detections, compute_map
import yaml

data_cfg = yaml.safe_load(open('/content/data_aug/data.yaml'))
class_names = data_cfg['names'] if isinstance(data_cfg['names'], dict) else dict(enumerate(data_cfg['names']))

model = YOLO('runs/detect/tooth_localization/weights/best.pt')
test_ds = sv.DetectionDataset.from_yolo(
    images_directory_path='/content/data_aug/test/images',
    annotations_directory_path='/content/data_aug/test/labels',
    data_yaml_path='/content/data_aug/data.yaml',
)

gts, preds = [], []
for path, image, annotation in test_ds:
    result = model(image, conf=0.25, verbose=False)[0]
    preds.append(sv.Detections.from_ultralytics(result))
    gts.append(annotation)

report = evaluate_detections(gts, preds, num_classes=len(class_names), class_names=class_names)
import pandas as pd
pd.DataFrame(report['per_class'])


## 7. Grad-CAM explainability demo

In [ ]:
import torch, cv2, matplotlib.pyplot as plt
from explainability.gradcam import gradcam_unet, overlay_heatmap
from models.segmentation.unet import UNet

ckpt = torch.load('models/segmentation/runs/segment/unet_best.pt', map_location='cpu')
unet = UNet(n_channels=1, n_classes=ckpt['num_classes'])
unet.load_state_dict(ckpt['model_state'])
unet.eval()

sample_path = test_ds.images[list(test_ds.images.keys())[0]]  # replace with any crop path
# ... load a tooth crop, run gradcam_unet(unet, tensor, target_class=1), overlay, plt.imshow(...)


## 8. Deploy API + Streamlit app (ngrok tunnel for Colab demo)

In [ ]:
!pip install -q pyngrok
from pyngrok import ngrok
import os

# Set your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken
ngrok.set_auth_token('YOUR_NGROK_TOKEN')

os.environ['DETECTOR_WEIGHTS'] = 'runs/detect/tooth_localization/weights/best.pt'
os.environ['SEGMENTER_WEIGHTS'] = 'models/segmentation/runs/segment/unet_best.pt'

get_ipython().system_raw('uvicorn api.main:app --host 0.0.0.0 --port 8000 &')
api_tunnel = ngrok.connect(8000)
print('API:', api_tunnel.public_url)


In [ ]:
os.environ['DENTAL_API_URL'] = api_tunnel.public_url
get_ipython().system_raw('streamlit run app/streamlit_app.py --server.port 8501 &')
app_tunnel = ngrok.connect(8501)
print('App:', app_tunnel.public_url)


## 9. (Optional) Push trained weights + metrics to your GitHub repo for the README

In [ ]:
# copy best.pt files, run_metadata.json, and the metrics report into results/
# then git add/commit/push from this Colab session or download and push locally
